# Multi-Model Probability Batch Pipeline

This notebook keeps the streamlined pipeline and adds scenario-based execution so you can switch folders, forcing files, burn layers, and cohesion settings without rewriting the workflow.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from landlab.components import FlowAccumulator, SinkFillerBarnes
from landlab.components.landslides import LandslideProbability
from landlab.grid.mappers import map_node_to_cell
from landlab.io import esri_ascii

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "notebook").exists():
    REPO_ROOT = REPO_ROOT.parent
NOTEBOOK_DIR = REPO_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from potential_evapotranspiration_field_OFFICIAL import PotentialEvapotranspiration
from radiation_field_OFFICIAL import Radiation
from soil_moisture_dynamics import SoilMoisture


In [ ]:
BASE_SETTINGS = {
    "number_of_iterations": 1000,
    "use_sink_fill": True,
    "ignore_sink_overfill": False,
    "dem_nodata": -999999.0,
    "field_nodata": {
        "soil__thickness": 2.549999999999999822e+00,
        "soil__density": 4.259999999999999898e-01,
        "soil__internal_friction_angle": 2.700000000000000000e+01,
        "soil__saturated_hydraulic_conductivity": 7.052563416309979523e+93,
        "vegetation__plant_functional_type": -9.999000000000000000e+03,
        "soil__maximum_total_cohesion": -9.999000000000000000e+03,
        "soil__mode_total_cohesion": -9.999000000000000000e+03,
        "soil__minimum_total_cohesion": -9.999000000000000000e+03,
    },
    "lai_by_pft": {0: 1.5, 1: 2.0, 2: 4.0, 3: 1.0},
    "cohesion_fields": (
        "soil__minimum_total_cohesion",
        "soil__mode_total_cohesion",
        "soil__maximum_total_cohesion",
    ),
    "latitude": 47.7,
    "albedo": 0.2,
    "zveg": 0.5,
    "z_wind": 2.0,
    "vwind": 3.04,
    "relative_humidity": 0.8,
    "lapse_rate_c_per_km": 4.5,
    "initial_time_years": 0.22,
    "storm_duration_hours": 24.0,
    "min_soil_thickness_m": 0.01,
    "min_transmissivity": 0.01,
    "recharge_floor": 0.01,
    "apply_cohesion_reduction": False,
    "cohesion_reduction_by_burn": {1: 0.00, 2: 0.15, 3: 0.35, 4: 0.60},
}

SCENARIO = {
    "name": "pioneer_cut1",
    "scenario_dir": Path("/mnt/c/Users/amehedi/Downloads/ml_debris/pioneer/output/cut1"),
    "precip_csv": Path("/mnt/c/Users/amehedi/Downloads/ml_debris/pioneer/output/cut1/precip_2026-01.csv"),
    "temp_csv": Path("/mnt/c/Users/amehedi/Downloads/ml_debris/pioneer/output/cut1/temp_2026-01.csv"),
}

SCENARIO_BATCH = [SCENARIO]


In [ ]:
def add_or_update_field(grid, name, values, at="node"):
    values = np.asarray(values)
    container = grid.at_node if at == "node" else grid.at_cell
    if name in container:
        container[name][:] = values
    else:
        grid.add_field(name, values.copy(), at=at, clobber=True)
    return container[name]


def load_ascii_values(path, field_name):
    with open(path) as f:
        temp_grid = esri_ascii.load(f, name=field_name)
    return temp_grid.at_node[field_name]


def load_node_field(grid, base_dir, filename, field_name, nodata_value=None, transform=None, dtype=None):
    raw = load_ascii_values(base_dir / filename, field_name)
    values = raw.copy()
    if transform is not None:
        values = transform(values)
    if dtype is not None:
        values = values.astype(dtype)
    add_or_update_field(grid, field_name, values, at="node")
    if nodata_value is not None:
        grid.set_nodata_nodes_to_closed(raw, nodata_value)
    return raw, grid.at_node[field_name]


def close_dem_nodata(grid, nodata_value):
    z = grid.at_node["topographic__elevation"]
    bad = np.isclose(z, nodata_value, atol=1e-2) | (z < -1e5)
    grid.status_at_node[bad] = grid.BC_NODE_IS_CLOSED
    z[bad] = 0.0
    return z


def load_forcing_arrays(precip_csv, temp_csv, elevation, z_ref, lapse_rate):
    precip = pd.read_csv(precip_csv, parse_dates=["datetime"]).sort_values("datetime")
    temp = pd.read_csv(temp_csv, parse_dates=["datetime"]).sort_values("datetime")
    forcing = precip.merge(temp, on="datetime", how="inner", validate="one_to_one")
    if forcing.empty:
        raise ValueError("No forcing rows were loaded")

    delta_z_km = (elevation - z_ref) / 1000.0
    rainfall_arrays = [np.full_like(elevation, p, dtype=float) for p in forcing["precip_mm"].to_numpy()]
    tempmin_arrays = [t - lapse_rate * delta_z_km for t in forcing["tmin_c"].to_numpy()]
    tempmax_arrays = [t - lapse_rate * delta_z_km for t in forcing["tmax_c"].to_numpy()]
    return forcing, rainfall_arrays, tempmin_arrays, tempmax_arrays


def cell_to_node(grid, cell_values, fill_value=0.0):
    out = np.full(grid.number_of_nodes, fill_value, dtype=float)
    out[grid.node_at_cell] = np.asarray(cell_values)
    return out


def patch_component_field_overwrite():
    def _overwrite_process_field(self, field, field_name):
        if isinstance(field, np.ndarray) and np.shape(field) == np.shape(self._grid.at_node["topographic__elevation"]):
            if field_name in self._gridCopy.at_node:
                self._gridCopy.at_node[field_name][:] = field
            else:
                self._gridCopy.add_field(field_name, field.copy(), at="node")
            return map_node_to_cell(self._gridCopy, field_name)
        return field

    Radiation._process_field = _overwrite_process_field
    PotentialEvapotranspiration._process_field = _overwrite_process_field


def update_cohesion_fields(grid, cohesion_fields, reduction_by_burn=None):
    burn = grid.at_node["burn__severity"].astype(int)
    mult = np.ones(grid.number_of_nodes, dtype=float)
    if reduction_by_burn is not None:
        for cls, red in reduction_by_burn.items():
            mult[burn == cls] = 1.0 - red

    for field in cohesion_fields:
        backup_field = f"{field}_pre"
        if backup_field not in grid.at_node:
            add_or_update_field(grid, backup_field, grid.at_node[field].copy(), at="node")
        values = grid.at_node[backup_field].copy() * mult
        add_or_update_field(grid, field, values, at="node")


In [ ]:
def run_pipeline(config, base_settings=BASE_SETTINGS):
    settings = dict(base_settings)
    settings.update(config)

    scenario_dir = Path(settings["scenario_dir"])
    precip_csv = Path(settings["precip_csv"])
    temp_csv = Path(settings["temp_csv"])

    with open(scenario_dir / "topographic__elevation.asc") as f:
        grid = esri_ascii.load(f, name="topographic__elevation")

    z = close_dem_nodata(grid, nodata_value=settings["dem_nodata"])
    outlet_id = grid.core_nodes[np.argmin(z[grid.core_nodes])]
    grid.set_watershed_boundary_condition_outlet_id(outlet_id, z)

    if settings["use_sink_fill"]:
        sink_filler = SinkFillerBarnes(
            grid,
            "topographic__elevation",
            method="D8",
            fill_flat=False,
            ignore_overfill=settings["ignore_sink_overfill"],
        )
        sink_filler.run_one_step()

    flow_accumulator = FlowAccumulator(
        grid,
        surface="topographic__elevation",
        flow_director="FlowDirectorD8",
        runoff_rate=None,
    )
    drainage_area, discharge = flow_accumulator.accumulate_flow()

    add_or_update_field(grid, "topographic__slope", grid.at_node["topographic__steepest_slope"], at="node")
    add_or_update_field(
        grid,
        "topographic__specific_contributing_area",
        drainage_area / grid.dx,
        at="node",
    )

    zmin = float(z[grid.core_nodes].min())

    field_nodata = settings["field_nodata"]
    load_node_field(grid, scenario_dir, "soil__thickness.asc", "soil__thickness", nodata_value=field_nodata["soil__thickness"], transform=lambda x: x / 100.0)
    load_node_field(grid, scenario_dir, "soil__density.asc", "soil__density", nodata_value=field_nodata["soil__density"])
    load_node_field(grid, scenario_dir, "soil__internal_friction_angle.asc", "soil__internal_friction_angle", nodata_value=field_nodata["soil__internal_friction_angle"])
    load_node_field(grid, scenario_dir, "porosity.asc", "porosity")
    load_node_field(grid, scenario_dir, "field__capacity.asc", "field__capacity")
    load_node_field(grid, scenario_dir, "wilting__point.asc", "wilting__point")
    load_node_field(grid, scenario_dir, "soil__saturated_hydraulic_conductivity.asc", "soil__saturated_hydraulic_conductivity", nodata_value=field_nodata["soil__saturated_hydraulic_conductivity"])
    load_node_field(grid, scenario_dir, "vegetation__plant_functional_type.asc", "vegetation__plant_functional_type", nodata_value=field_nodata["vegetation__plant_functional_type"], dtype=int)
    load_node_field(grid, scenario_dir, "soil__maximum_total_cohesion.asc", "soil__maximum_total_cohesion", nodata_value=field_nodata["soil__maximum_total_cohesion"])
    load_node_field(grid, scenario_dir, "soil__mode_total_cohesion.asc", "soil__mode_total_cohesion", nodata_value=field_nodata["soil__mode_total_cohesion"])
    load_node_field(grid, scenario_dir, "soil__minimum_total_cohesion.asc", "soil__minimum_total_cohesion", nodata_value=field_nodata["soil__minimum_total_cohesion"])
    load_node_field(grid, scenario_dir, "burn__severity.asc", "burn__severity")

    open_nodes = grid.status_at_node != grid.BC_NODE_IS_CLOSED
    hs = grid.at_node["soil__thickness"]
    hs[(open_nodes) & (hs <= 0)] = settings["min_soil_thickness_m"]

    ksat = grid.at_node["soil__saturated_hydraulic_conductivity"]
    transmissivity = ksat * 2.5 * hs
    transmissivity[transmissivity <= 0] = settings["min_transmissivity"]
    add_or_update_field(grid, "soil__transmissivity", transmissivity, at="node")

    pft = grid.at_node["vegetation__plant_functional_type"].astype(int)
    pft[pft < 0] = 0
    grid.at_node["vegetation__plant_functional_type"][:] = pft
    add_or_update_field(grid, "vegetation__plant_functional_type", pft[grid.node_at_cell].astype(int), at="cell")

    lai = np.full(grid.number_of_nodes, settings["lai_by_pft"][3], dtype=float)
    for cls, value in settings["lai_by_pft"].items():
        lai[pft == cls] = value
    add_or_update_field(grid, "vegetation__live_leaf_area_index", lai, at="node")
    add_or_update_field(grid, "vegetation__cover_fraction", lai / 4.0, at="node")
    add_or_update_field(grid, "vegetation__live_leaf_area_index", lai[grid.node_at_cell], at="cell")
    add_or_update_field(grid, "vegetation__cover_fraction", (lai / 4.0)[grid.node_at_cell], at="cell")

    burn = grid.at_node["burn__severity"].astype(int)
    burn[~np.isin(burn, [2, 3, 4])] = 1
    grid.at_node["burn__severity"][:] = burn

    initial_saturation = (
        0.5 * (grid.at_node["field__capacity"] - grid.at_node["wilting__point"])
        + grid.at_node["wilting__point"]
    ) / grid.at_node["porosity"]
    add_or_update_field(grid, "soil_moisture__initial_saturation_fraction", initial_saturation, at="node")
    add_or_update_field(grid, "soil_moisture__initial_saturation_fraction", initial_saturation[grid.node_at_cell], at="cell")
    add_or_update_field(grid, "saturated__hydraulic_conductivity", ksat[grid.node_at_cell], at="cell")
    add_or_update_field(grid, "rainfall__daily_depth", np.zeros(grid.number_of_cells, dtype=float), at="cell")

    reduction = settings["cohesion_reduction_by_burn"] if settings["apply_cohesion_reduction"] else None
    update_cohesion_fields(grid, settings["cohesion_fields"], reduction_by_burn=reduction)

    forcing, rainfall_arrays, tempmin_arrays, tempmax_arrays = load_forcing_arrays(
        precip_csv,
        temp_csv,
        z,
        zmin,
        settings["lapse_rate_c_per_km"],
    )

    patch_component_field_overwrite()
    pet = PotentialEvapotranspiration(grid, method="PenmanMonteith")
    soil_moisture = SoilMoisture(grid)

    pet._latitude = settings["latitude"]
    pet._a = settings["albedo"]
    pet._zm = settings["z_wind"]
    pet._zveg = settings["zveg"] * np.ones(grid.number_of_cells)
    pet._vz = settings["vwind"] * np.ones(grid.number_of_cells)
    pet._relative_humidity = settings["relative_humidity"] * np.ones(grid.number_of_cells)
    pet._LAI = grid.at_node["vegetation__live_leaf_area_index"]

    soil_moisture._Tb = settings["storm_duration_hours"]

    current_time = settings["initial_time_years"]
    node_at_cell = grid.node_at_cell
    runoff_arrays = []
    recharge_arrays = []
    soil_moisture_arrays = []
    et_arrays = []

    for rainfall, tempmin, tempmax in zip(rainfall_arrays, tempmin_arrays, tempmax_arrays):
        add_or_update_field(grid, "Precipitation", rainfall, at="node")
        add_or_update_field(grid, "Tmin", tempmin, at="node")
        add_or_update_field(grid, "Tmax", tempmax, at="node")
        grid.at_cell["rainfall__daily_depth"][:] = rainfall[node_at_cell]

        pet._current_time = current_time
        pet._Tmin = tempmin
        pet._Tmax = tempmax
        pet.update()

        soil_moisture._current_time = current_time
        soil_moisture.update()

        recharge_arrays.append(cell_to_node(grid, grid.at_cell["soil_moisture__root_zone_leakage"]))
        runoff_arrays.append(cell_to_node(grid, grid.at_cell["surface__runoff"]))
        soil_moisture_arrays.append(cell_to_node(grid, grid.at_cell["soil_moisture__saturation_fraction"]))
        et_arrays.append(cell_to_node(grid, grid.at_cell["surface__evapotranspiration"]))

        current_time = soil_moisture.current_time

    mean_runoff = np.mean(runoff_arrays, axis=0)
    mean_recharge = np.mean(recharge_arrays, axis=0)
    max_runoff = np.maximum.reduce(runoff_arrays)
    max_recharge = np.maximum.reduce(recharge_arrays)

    r = max_recharge.copy()
    r[r <= 0] = settings["recharge_floor"]
    rstd = r * 0.1

    add_or_update_field(grid, "groundwater__recharge_mean", r, at="node")
    add_or_update_field(grid, "groundwater__recharge_standard_deviation", rstd, at="node")
    add_or_update_field(grid, "groundwater__runoff_mean", max_runoff, at="node")
    add_or_update_field(grid, "test_runoff", mean_runoff, at="node")
    add_or_update_field(grid, "test_recharge", mean_recharge, at="node")

    ls_prob = LandslideProbability(
        grid,
        number_of_iterations=settings["number_of_iterations"],
        groundwater__recharge_distribution="lognormal_spatial",
        groundwater__recharge_mean=r,
        groundwater__recharge_standard_deviation=rstd,
    )
    ls_prob.calculate_landslide_probability()

    summary = {
        "name": settings["name"],
        "days": int(len(forcing)),
        "core_nodes": int(grid.number_of_core_nodes),
        "mean_recharge_core": float(np.mean(mean_recharge[grid.core_nodes])),
        "mean_runoff_core": float(np.mean(mean_runoff[grid.core_nodes])),
        "mean_failure_core": float(np.mean(grid.at_node["landslide__probability_of_failure"][grid.core_nodes])),
        "max_failure_core": float(np.max(grid.at_node["landslide__probability_of_failure"][grid.core_nodes])),
    }

    results = {
        "forcing": forcing,
        "mean_runoff": mean_runoff,
        "mean_recharge": mean_recharge,
        "max_runoff": max_runoff,
        "max_recharge": max_recharge,
        "runoff_arrays": runoff_arrays,
        "recharge_arrays": recharge_arrays,
        "soil_moisture_arrays": soil_moisture_arrays,
        "ET_arrays": et_arrays,
        "summary": summary,
    }

    return grid, results


def run_many(configs, keep_grids=False):
    outputs = {}
    summaries = []
    for config in configs:
        grid, results = run_pipeline(config)
        record = {"results": results}
        if keep_grids:
            record["grid"] = grid
        outputs[config["name"]] = record
        summaries.append(results["summary"])
    return outputs, pd.DataFrame(summaries)


## Single Scenario Run

Run the active scenario and keep the grid plus aggregated outputs in memory.

In [ ]:
grid, results = run_pipeline(SCENARIO)
summary = pd.Series(results["summary"])
summary


## Batch Run

Populate `SCENARIO_BATCH` with multiple scenario dictionaries and run them without keeping all grids in memory.

In [ ]:
batch_outputs, batch_summary = run_many(SCENARIO_BATCH, keep_grids=False)
batch_summary
